# 4. Inference

In [ ]:
# Main imports
import numpy as np
import sys
from pathlib import Path
import json

# Scikit-learn imports
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, ShuffleSplit

# Add the project root to the Python path
PROJECT_ROOT = Path().resolve().parents[1]
sys.path.append(str(PROJECT_ROOT))

# Import custom modules
from src.config import PROJECT_ROOT, DATA_ROOT, initialize_h5
from src.utilities import (
    read_coord_transform_results,
    save_inference_results,
    get_keys_h5,
    get_attrs_h5,
    repack_h5,
)
from src.analysis import generate_time_intervals
from src.modeling import (SpringPendulumRegressor, compute_derived_params,
                          print_inference_results, store_interval_inference_results,
                          initialize_inference_results, finalize_inference_results)
from src.validation import generate_bootstrap_samples, compute_bootstrap_statistics

In [ ]:
# Directory and files setup

TYPE = 'tangle_00'
STAGE = '04_inference'

INPUT_PATH = DATA_ROOT / TYPE / '03_coord_transform.h5'
OUTPUT_PATH = DATA_ROOT / TYPE / (STAGE + '.h5')

print('INPUT_PATH: \t{:}'.format(INPUT_PATH))
print('OUTPUT_PATH: \t{:}'.format(OUTPUT_PATH))
print()

# Initialize output file
overwrite = True
verbose = True

initialize_h5(OUTPUT_PATH, overwrite=overwrite, verbose=verbose)

# Time interval and coordinate limit configuration
attrs = get_attrs_h5(INPUT_PATH)
time_interval_config = json.loads(attrs['time_interval_config'])
coord_limit_config = json.loads(attrs['coord_limit_config'])

## Inference of Spring Pendulum Dynamics

### Configuration

In [ ]:
# Inference configuration

equation = 'z_equation'
constrain_L0_monotonic = False
constrain_L0_value = 'bootstrap_mins'   # options: 'bootstrap_mins', 'bootstrap_ci_lowers'

# Initialize regressor
# Velocity-dependent terms are excluded from this fit.
regressor_config = {
    'breakpoint_value': 0,
    'fix_gravity': True,
    'g_value': 9.81,
    'half_spring': True,
    'use_features': ['constant', 
                     'cos_theta', 
                     'r_cos_theta', 
                     ],
    'constrain_positive': True,
    'L0_min': None
}
spring_pendulum_regressor = SpringPendulumRegressor(**regressor_config)

# Initialize grid search
cv_config = {
    'param_grid': {
        'breakpoint_value': np.linspace(0.02, 0.06, 25) if regressor_config['half_spring'] else [0],
    },
    'cv': ShuffleSplit(n_splits=10, test_size=0.2, random_state=0),
    'scoring': 'neg_mean_squared_error',
    'n_jobs': -1,
}
spring_pendulum_cv = GridSearchCV(spring_pendulum_regressor, **cv_config)

# Bootstrap parameters
bootstrap_config = {
    'n_bootstrap': 100,
    'random_state': 0
}

# Time interval config
fit_sample = False   # If true, only fit the model on the sampletime interval

if verbose:
    print('Regressor')
    print('-'*60)
    for param, value in spring_pendulum_regressor.get_params().items():
        print(f'{param}: {value}')
    print()
    print('Cross_validation')
    print('-'*60)
    for param in ['scoring', 'cv', 'n_jobs']:
        print(f'{param}: {getattr(spring_pendulum_cv, param)}')
    print()
    print('Bootstrap')
    print('-'*60)
    print(f'n_bootstrap: {bootstrap_config["n_bootstrap"]}')
    print(f'random_state: {bootstrap_config["random_state"]}')
    print()
    print('Time intervals')
    print('-'*60)
    print(f'{json.dumps(time_interval_config, indent=2)[:100]}\n...')

### Inference loop over time intervals

In [ ]:
# Inference loop over time intervals

data_labels = get_keys_h5(INPUT_PATH)

# Parameter extraction function for bootstrap (does not depend on loop state)
extract_spring_params = lambda estimator: estimator.get_physical_parameters()[0]

for data_label in data_labels:

    # 1. Read regression data and experiment metadata
    coord_datasets, params, coord_transform_config, equations_data, candidates_data = read_coord_transform_results(INPUT_PATH, data_label)
    equation_data = equations_data[equation]
    feature_data, feature_names, feature_labels, feature_df = (equation_data[key] for key in ['feature_data', 'feature_names', 'feature_labels', 'feature_df'])
    target_data, target_name, target_label, target_df = (equation_data[key] for key in ['target_data', 'target_name', 'target_label', 'target_df'])
    Omega = 2 * np.pi * params['piston_frequency']   # driving angular frequency (rad/s)

    # 2. Generate experiment-specific time intervals and param_grid
    interval_params = time_interval_config[data_label]
    interval_params['time_range'][0] = max(interval_params['time_range'][0], coord_datasets['t'].min())
    interval_params['time_range'][1] = min(interval_params['time_range'][1], coord_datasets['t'].max())
    
    time_intervals = generate_time_intervals(
        interval_params['time_range'], 
        interval_params['interval_sep'], 
        interval_params['interval_len'],
        sample=fit_sample,  # Set to True if you want only sample_interval
        sample_interval=interval_params.get('sample_interval', []),
    )

    r_min, r_max = feature_df['r'].min(), feature_df['r'].max()
    if regressor_config['half_spring']:
        n_grid = spring_pendulum_cv.param_grid['breakpoint_value'].shape[0]
        spring_pendulum_cv.param_grid['breakpoint_value'] = np.linspace(r_min + 0.001, r_max - 0.001, n_grid)

    if verbose:
        print('=' * 100)
        print(f"{data_label}: fitting {len(time_intervals)} time intervals")

    # 3. Initialize results accumulator
    inference_results = initialize_inference_results(time_intervals)

    # 4. Inference for each time interval
    for i, (t_start, t_end) in enumerate(time_intervals):

        # a. Filter data to interval
        interval_mask = (coord_datasets['t'] >= t_start) & (coord_datasets['t'] <= t_end)
        feature_df_interval = feature_df[interval_mask].reset_index(drop=True)
        target_df_interval = target_df[interval_mask].reset_index(drop=True)

        # b. Update L0_min for monotonic constraint (if enabled)
        if constrain_L0_monotonic:
            if i == 0:
                L0_min_value = None
            else:
                L0_idx = inference_results['spring_pendulum_params']['param_names'].index('L0')
                L0_min_value = inference_results['spring_pendulum_params'][constrain_L0_value][-1][L0_idx]
            spring_pendulum_cv.estimator.L0_min = L0_min_value

        # c. Fit regressor and grid search over parameters
        spring_pendulum_cv.fit(feature_df_interval, target_df_interval)

        # d. Extract physical parameters and compute derived params
        spring_pendulum_optimal_regressor = spring_pendulum_cv.best_estimator_
        spring_pendulum_params, spring_pendulum_param_labels = spring_pendulum_optimal_regressor.get_physical_parameters()
        derived_params, derived_param_labels = compute_derived_params(spring_pendulum_params, Omega=Omega)
        spring_pendulum_params.update(derived_params)
        spring_pendulum_param_labels.update(derived_param_labels)

        # e. Bootstrap uncertainty estimation
        bootstrap_samples = generate_bootstrap_samples(
            estimator=spring_pendulum_optimal_regressor, 
            X=feature_df_interval, 
            y=target_df_interval, 
            extract_params=extract_spring_params, 
            **bootstrap_config)
        for b, params_bootstrap in enumerate(bootstrap_samples):
            derived_params_bootstrap, _ = compute_derived_params(params_bootstrap, Omega=Omega)
            bootstrap_samples[b].update(derived_params_bootstrap)
        bootstrap_statistics = compute_bootstrap_statistics(bootstrap_samples)
        for param, value in spring_pendulum_params.items():
            spring_pendulum_params[param] = (bootstrap_statistics[param]['mean'], bootstrap_statistics[param])

        # f. Compute validation metrics
        target_predict = spring_pendulum_optimal_regressor.predict(feature_df_interval)
        validation = {
            'cv_best_params': spring_pendulum_cv.best_params_,
            'cv_param_grid': np.array(spring_pendulum_cv.cv_results_[f'param_{list(spring_pendulum_cv.best_params_.keys())[0]}'], dtype=float),
            'cv_mean_test_score': -spring_pendulum_cv.cv_results_['mean_test_score'],
            'cv_std_test_score': spring_pendulum_cv.cv_results_['std_test_score'],
            'cv_mean_train_score': -spring_pendulum_cv.cv_results_.get('mean_train_score', np.array([])),
            'cv_std_train_score': spring_pendulum_cv.cv_results_.get('std_train_score', np.array([])),
            'mse_score': mean_squared_error(target_df_interval, target_predict),
            'r2_score': r2_score(target_df_interval, target_predict)
        }

        # g. Store results for this interval
        store_interval_inference_results(inference_results, spring_pendulum_params, spring_pendulum_param_labels, validation)

        # h. Print results for this interval
        if verbose:
            interval_string = f"Time interval ({i+1}/{len(time_intervals)}): [{t_start:.1f}, {t_end:.1f}]"
            print_inference_results(params, None, spring_pendulum_params, validation=validation, fmt='.5g', extra=interval_string)

    # 5. Finalize and save results
    finalize_inference_results(inference_results)
    save_inference_results(
        data_path=OUTPUT_PATH,
        data_label=data_label,
        data_params=params,
        regressor_config=regressor_config,
        cv_config=cv_config,
        bootstrap_config=bootstrap_config,
        inference_results=inference_results,
        equation=equation,
        hdf5_options={'compression': 'gzip', 'compression_opts': 9}
    )

repack_h5(OUTPUT_PATH, in_place=True)